import necessary libraries

In [ ]:
import pandas as pd
import numpy as np

Load raw dataset

In [ ]:
file = r"C:\Users\Lenovo\OneDrive\Documents\DataSprint\data\DataWave.csv"

df = pd.read_csv(file)


Inspect dataset


In [ ]:
df.isnull().sum()

df.info()
df.describe()
df.head()
df.tail()
df.columns

cleaning user_id

In [ ]:
df['user_id'].isnull().sum()
df = df.drop_duplicates(subset=['user_id'])

df= df.dropna(subset=['user_id'])


Convert all categorical text column to lower case and strip whitespaces. These are for the rows of each their respective columns. Ensure column is treated as text with the help of .astype(str).

In [ ]:
cat_cols = ['country', 'gender', 'subscription_type', 'churned']
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

Cleaning gender values

In [ ]:
df['gender'] = df['gender'].replace({
    'm': 'male',
    'f': 'female',
    'femle' : 'female',
    'female ': 'female',
    'others': 'other',
    'other ': 'other',
    'prefer not to say': 'unknown',
    'nan': np.nan
})

Cleaning churned columns. Convert yes/no or text to 0/1 integer  [user left => 1, user stayed => 0]

In [ ]:
mapping = {
    'yes': 1, 'y': 1, '1': 1, 'true': 1,
    'no': 0, 'n': 0, '0': 0, 'false': 0
}

df['churned'] = (
    df['churned']
    .astype(str).str.strip().str.lower()
    .map(mapping)
)


df['churned'].mean()   # Finding the mean of churned

Cleaning(Fix) Subscription Type

In [ ]:
df['subscription_type'] = df['subscription_type'].replace({
    'premuim': 'premium',
    'premum': 'premium',
    'free trial': 'free',
    'fam': 'family',
    'stud': 'student',
    'studnt': 'student'
})

df['subscription_type'].value_counts()   # Counting the number of each subscription type

Cleaning country

In [ ]:
df['country'] = df['country'].replace({
    'u.k.': 'United kingdom',
    'uk': 'United Kingdom',
    'U.K': ' United Kingdom',
    'united kingdom': 'United Kingdom',
    'United kingdom': 'United Kingdom',
    'ind': 'India',
    'india': 'India',
    'usa': 'United States',
    'us': 'United States',
    'nepal': 'Nepal',
    'nigeria': 'Nigeria',
    'ghana': 'Ghana',
    'kenya': 'Kenya',
    'brazil': 'Brazil',
    'south africa': 'South Africa'
})
df['country'].value_counts()

Clean numeric columns. Convert invalid text like "ten" to NaN

In [ ]:
num_cols = ['age', 'avg_listening_hours_per_week', 'total_songs_played', 'skip_rate', 'satisfaction_score', 'monthly_fee']

for cols in num_cols:
    df[cols] = pd.to_numeric(df[cols], errors='coerce')   # convert invalid values to NaN)


df['avg_listening_hours_per_week'].describe()   # Summary statistics of average listening hours per week

CLEAN join_date. Convert to datetime.

In [ ]:
df['join_date'] = pd.to_datetime(df['join_date'], format='%m/%d/%Y', errors='coerce')
df['join_date'].isnull().sum()

Checking Missing values

In [ ]:
df.isnull().sum()

Satisfaction score description.

In [ ]:
df['satisfaction_score'].describe()

IMPUTE MISSING VALUES PROPERLY [Fill in the missing data in the correct or appropriate way.]
1. Gender (leave as NaN or fill with "unknown"). [fillna means to fill in missing values]

In [ ]:
df['gender'] = df['gender'].fillna('unknown')

2. When analyzing the skip rate, it's important to use the median rather than the mean. This is because the skip rate data is often skewed, meaning it can have outliers that disproportionately affect the average. The median provides a more accurate representation of the typical skip rate in such cases.

In [ ]:
df['skip_rate'] = df['skip_rate'].fillna(df["skip_rate"].median())

3. Fill missing monthly fee values using the median value specific to each subscription type not the overall median.

In [ ]:
df['monthly_fee'] = (
    df.groupby('subscription_type')['monthly_fee'].transform(lambda x: x.fillna(x.median()))
    )

4. Fill missing values for join_date

In [ ]:
df['join_date'] = df['join_date'].fillna(df['join_date'].mode()[0])

Remove or fix numeric outliers:
1. skip_rate must be between 0 and 1 

In [ ]:
df = df[(df['skip_rate'] >= 0) & (df['skip_rate'] <= 1)]

2. Age should be between 10 and 100

In [ ]:
df = df[(df['age'] >= 10) & (df['age'] <= 100)]

3. Max listening hours per week = 168 (24*7)

In [ ]:
df = df[df['avg_listening_hours_per_week'] <= 168]   

4. Satisfaction must be between 1 and 10

In [ ]:
df = df[(df['satisfaction_score'] >= 1) & (df['satisfaction_score'] <= 10)]

5. Songs played cannot be negative + cap unrealistic values

In [ ]:
df = df[df['total_songs_played'] >= 0]
df = df[df['total_songs_played'] < df['total_songs_played'].quantile(0.999)]

6.  Monthly fee cannot be negative or extremely high

In [ ]:
df = df[df['monthly_fee'] >= 0]
df = df[df['monthly_fee'] < df['monthly_fee'].quantile(0.999)]

Reset index after removing rows

In [ ]:
df = df.reset_index(drop=True)

FINAL MISSING VALUE CHECK

In [ ]:
print("Final missing values:")
print(df.isnull().sum())
print(df.describe())
print(df.info())

cleaned dataset

In [ ]:
cleaned_dataset = r"C:\Users\Lenovo\OneDrive\Documents\DataSprint\data\cleaned_DataWave.csv"
df.to_csv(cleaned_dataset, index=False)

df = pd.read_csv(cleaned_dataset)

Relationship insights
1. Does churn differ by subscription type?(Churn rate by subscription type)

In [ ]:
df.groupby('subscription_type')['churned'].mean().sort_values(ascending=False)

2. Do heavy listeners churn less? (Listening hours: churned vs active). Low hours = more churn, High hours = more engagement

In [ ]:
df.groupby('churned')['avg_listening_hours_per_week'].mean()

3. Does satisfaction predict churn?(Satisfaction score: churned vs active). If churned users have lower scores then obvious red flag for company. 


In [ ]:
df.groupby('churned')['satisfaction_score'].mean()

4. Which country or region has highest churn? (Top 10 countries with highest churn)

In [ ]:
df.groupby('country')['churned'].mean().sort_values(ascending=False).head(10)